# BONUS: Advanced Streaming Patterns

This notebook covers four advanced patterns for incremental ingestion: stream-static joins, streaming aggregations with watermarking, error handling with `badRecordsPath`, and Change Data Feed for incremental ETL.

> **Note:** This notebook extends <a href="$./04_incremental_ingestion">04 — Incremental Ingestion</a>, but it runs **standalone**. The setup cell below recreates the inputs it needs: the customers table, a landing directory with order batches and the Auto Loader stream. All streams write to Delta tables with `trigger(availableNow=True)`, so they run on serverless and classic compute. Memory and console sinks and `processingTime` triggers work only on classic compute.

In [0]:
%run ../../setup/00_setup

### Configuration

Import the libraries. The setup cell after it defines the paths for source data, checkpoints and schema locations.

In [0]:
# Libraries used by all demos in this notebook
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime
import time

### Standalone Setup

The setup defines the same names that 04 uses (`TABLE_CUSTOMERS`, `stream_batches`, `STREAM_SOURCE_PATH`, `CHECKPOINT_BASE_PATH`, …) in a separate demo folder. This way the notebook does not depend on the state that 04 leaves behind.

In [0]:
# Standalone setup: recreates what 04_incremental_ingestion defines
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {BRONZE_SCHEMA}")

DEMO_BASE_PATH       = f"{DATASET_PATH}/ingestion_demo_bonus"
BATCH_SOURCE_PATH    = f"{DEMO_BASE_PATH}/batch_source"
STREAM_SOURCE_PATH   = f"{DEMO_BASE_PATH}/stream_source"
CHECKPOINT_BASE_PATH = f"{DEMO_BASE_PATH}/checkpoints"
SCHEMA_BASE_PATH     = f"{DEMO_BASE_PATH}/schemas"
BAD_RECORDS_PATH     = f"{DEMO_BASE_PATH}/bad_records"
dbutils.fs.rm(DEMO_BASE_PATH, True)   # fresh start: sources, checkpoints and schemas

# Static dimension: the customers table that 04 loads with COPY INTO, created here if missing
TABLE_CUSTOMERS = f"{BRONZE_SCHEMA}.customers_batch"
if not spark.catalog.tableExists(f"{CATALOG}.{TABLE_CUSTOMERS}"):
    (spark.read.option("header", "true").csv(f"{DATASET_PATH}/customers/customers.csv")
        .write.saveAsTable(TABLE_CUSTOMERS))
    print(f"Created {TABLE_CUSTOMERS} from customers.csv")

# Split the stream files into 20 order batches that "arrive" during the demos
df_all_orders = spark.read.json(f"{DATASET_PATH}/orders/stream/*.json")
stream_batches = df_all_orders.randomSplit([0.05] * 20, seed=42)
stream_batches[0].write.mode("overwrite").json(f"{STREAM_SOURCE_PATH}/batch_01")

def enriched_orders_stream(name):
    """Auto Loader stream over the landing directory + metadata columns.

    One schema location per query."""
    return (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", f"{SCHEMA_BASE_PATH}/{name}")
        .option("cloudFiles.inferColumnTypes", "true")
        .load(STREAM_SOURCE_PATH)
        .withColumn("_processing_time", F.current_timestamp())
        .withColumn("_source_file", F.col("_metadata.file_path"))
    )

df_enriched = enriched_orders_stream("join")
print(f"Setup ready. Landing directory: {STREAM_SOURCE_PATH} (batch_01 present)")

## Stream-Static Joins & Aggregations

> **Summary** (see the [Structured Streaming programming guide](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html)): Structured Streaming is a scalable, fault-tolerant stream processing engine built on the Spark SQL engine. It treats a live data stream as a table that keeps growing. You write the query as for a static table, and Spark runs it incrementally.

A stream-static join enriches a stream (for example, orders) with a static table (for example, customers). Spark reads the static table again at the start of each micro-batch.

The static side needs a unique join key. `customers.csv` repeats some `customer_id` values, and the next cell counts them. A `LEFT JOIN` on the raw customers would return one row per match for every order of those customers. So the next cell removes the duplicates first. Later, the cell that counts orders without a customer match (after the second run) ends with a `# Check` that confirms one joined row per order.

In [0]:
# Count customer_id values that occur more than once: each one would multiply joined orders
dup_ids = (spark.table(TABLE_CUSTOMERS).where("customer_id IS NOT NULL")
           .groupBy("customer_id").count().where("count > 1").count())
print(f"customer_id values that occur more than once: {dup_ids:,}")

# Static dimension with one row per customer_id (dropDuplicates keeps an arbitrary row per id)
df_static_customers = spark.table(TABLE_CUSTOMERS).dropDuplicates(["customer_id"])
display(df_static_customers)

The printed line shows how many `customer_id` values repeat, and the table shows the deduplicated customers.

### Demo: Stream-Static Join (Append Mode)

The join writes enriched orders to a Delta table, as a real pipeline would. The first cell drops the target table, so a re-run starts from an empty table.

In [0]:
# Drop the target table so a re-run starts clean
TABLE_JOINED = f"{CATALOG}.{BRONZE_SCHEMA}.enriched_orders"
spark.sql(f"DROP TABLE IF EXISTS {TABLE_JOINED}")

In [0]:
# Temp views let the join run as SQL; LEFT JOIN keeps orders without a customer
df_enriched.createOrReplaceTempView("enriched_orders_stream")
df_static_customers.createOrReplaceTempView("static_customers")

df_joined = spark.sql("""
SELECT
  o.order_id,
  o.total_amount,
  c.first_name,
  c.last_name,
  c.email,
  o._processing_time
FROM enriched_orders_stream o
LEFT JOIN static_customers c
  ON o.customer_id = c.customer_id
""")

In [0]:
# Write to a Delta table, because memory and console sinks are not available on serverless
def run_join_stream():
    q = (df_joined.writeStream
        .format("delta")
        .outputMode("append")   # stream-static joins are append-only
        .option("checkpointLocation", f"{CHECKPOINT_BASE_PATH}/join_demo")
        .trigger(availableNow=True)
        .toTable(TABLE_JOINED)
    )
    q.awaitTermination()
    print(f"{TABLE_JOINED}: {spark.table(TABLE_JOINED).count():,} rows")

run_join_stream()   # processes batch_01

In [0]:
# Batches 2-3 arrive, then the next scheduled run picks up only those files
for i in range(1, 3):
    batch_num = i + 1
    stream_batches[i].write.mode("overwrite").json(f"{STREAM_SOURCE_PATH}/batch_{batch_num:02d}")
    print(f"Arrived: batch_{batch_num:02d}")

run_join_stream()
# Enriched rows with a customer match; the next cell counts orders without one
display(spark.table(TABLE_JOINED).filter("first_name IS NOT NULL").limit(10))

In [0]:
# Orders without a matching customer keep NULL customer columns (LEFT JOIN)
display(spark.sql(f"""
    SELECT count(*) AS total_rows,
           count_if(first_name IS NULL) AS without_customer_match
    FROM {TABLE_JOINED}
"""))

# One joined row per landed order means no fan-out from duplicate customers
landed = spark.read.json(f"{STREAM_SOURCE_PATH}/batch_0[1-3]").count()
joined = spark.table(TABLE_JOINED).count()
print(f"Joined rows: {joined:,}, landed orders (batches 1-3): {landed:,}")

In the result, `without_customer_match` shows how many orders found no customer, and the `Joined rows` line should show the same number as the landed orders (one joined row per landed order).

### Demo: Streaming Aggregation (Event-Time Windows + Watermarking)

Windows and watermarks use **event time**: the time when the order happened (`order_datetime`), not the time when Spark processed it. A `current_timestamp()` column can never be late, so a watermark on processing time would never drop anything.

- `window("order_ts", "1 day")` creates tumbling 1-day windows by event time.
- `withWatermark("order_ts", "2 days")` makes Spark track the max event time seen. After each micro-batch, the watermark becomes *max event time − 2 days*. Rows for windows that ended before the watermark are **dropped as late**, and Spark removes their state. This keeps stateful aggregations bounded.
- Rows with a NULL timestamp or one outside the data period (orders from December 2024 and January 2025) are filtered out before the watermark. One outlier such as `2026-02-05` would push the watermark far ahead, and Spark would treat every later order as late.

Aggregations have three output modes:

- `update` emits only the windows that changed in a micro-batch, each with its new cumulative count.
- `append` emits each window once, after the watermark closes it.
- `complete` emits the whole result again.

A Delta table sink accepts only append and complete. So the demo writes the update-mode results with `foreachBatch`, which works with `availableNow` on serverless and classic compute.

> The landing batches are random samples of 12 days of orders. After the first run, the watermark sits about 2 days behind the newest order. So most orders in batches 4-5 are late, and Spark drops them. The counts at the end of the demo show how many.
>
> The `batch_id` column is the Structured Streaming micro-batch id, not the landing batch number. One run can write several micro-batches, and a micro-batch that changes no window adds no rows. So the ids can have gaps and can differ between runs.

In [0]:
# [1/2] Count orders per payment method in 1-day windows on event time
def with_event_time(df):
    """Parse order_datetime into an event-time column.

    Drop rows that cannot carry a watermark."""
    return (df
        .withColumn("order_ts", F.to_timestamp("order_datetime"))
        .filter(F.col("order_ts").isNotNull())
        # an outlier outside the data period (e.g. 2026-02-05) would push the watermark far ahead
        .filter(F.col("order_ts") < F.lit("2026-01-01").cast("timestamp"))
    )

windowed_counts = (with_event_time(enriched_orders_stream("agg"))
    # events more than 2 days behind the max event time are late
    .withWatermark("order_ts", "2 days")
    .groupBy(
        F.window("order_ts", "1 day"),
        "payment_method"
    )
    .count()
)

In [ ]:
# [2/2] Write in update mode through foreachBatch, because a Delta sink accepts only append or
# complete
TABLE_WINDOW_COUNTS = f"{CATALOG}.{BRONZE_SCHEMA}.orders_window_counts"
spark.sql(f"DROP TABLE IF EXISTS {TABLE_WINDOW_COUNTS}")

def write_window_updates(batch_df, batch_id):
    # batch_df holds only the windows updated in this micro-batch, with their cumulative counts
    (batch_df.withColumn("batch_id", F.lit(batch_id))
        .write.mode("append").saveAsTable(TABLE_WINDOW_COUNTS))

def run_agg_stream():
    q = (windowed_counts.writeStream
        .outputMode("update")
        .foreachBatch(write_window_updates)
        .option("checkpointLocation", f"{CHECKPOINT_BASE_PATH}/agg_demo")
        .trigger(availableNow=True)
        .start()
    )
    q.awaitTermination()

run_agg_stream()   # processes batches 1-3

In [0]:
# Batches 4-5 arrive. The next run emits only the windows that changed
# and drops orders older than the watermark that the previous run set
for i in range(3, 5):
    batch_num = i + 1
    stream_batches[i].write.mode("overwrite").json(f"{STREAM_SOURCE_PATH}/batch_{batch_num:02d}")
    print(f"Arrived: batch_{batch_num:02d}")

run_agg_stream()
display(spark.table(TABLE_WINDOW_COUNTS).orderBy("batch_id", "window", "payment_method"))

In [0]:
# Update mode re-emits a window with its new cumulative count every time it changes,
# so one (window, payment_method) can appear in several batch_ids: never sum counts across batches.
display(spark.sql(f"""
    SELECT batch_id, count(*) AS windows_emitted,
           min(window.start) AS earliest_window, max(window.start) AS latest_window
    FROM {TABLE_WINDOW_COUNTS}
    GROUP BY batch_id ORDER BY batch_id
"""))

# Current result = latest emitted row per (window, payment_method)
final_counts = spark.sql(f"""
    SELECT window, payment_method, count, batch_id
    FROM {TABLE_WINDOW_COUNTS}
    QUALIFY row_number() OVER (PARTITION BY window.start, payment_method ORDER BY batch_id DESC) = 1
""")
counted = final_counts.agg(F.sum("count")).first()[0]

# Everything that landed, with the same event-time filter; the difference is what the watermark
# dropped
landed = with_event_time(
    spark.read.option("recursiveFileLookup", "true").json(STREAM_SOURCE_PATH)
).count()
print(f"Orders with a valid event time landed (batches 1-5): {landed:,}")
print(f"Orders counted in the final windows:                 {counted:,}")
print(f"Dropped as late data (behind the watermark):          {landed - counted:,}")
display(final_counts.orderBy("window", "payment_method"))

# outputMode("append") would emit each window only once, after the watermark passes its end,
# so the "latest row" logic would not be needed.
# Alternative on classic compute only: .trigger(processingTime="5 seconds") keeps the query
# running until query.stop().

The three printed lines show how many orders landed, how many the final windows count and how many the watermark dropped as late.

### Demo: Malformed Records (`_corrupt_record` vs `badRecordsPath`)

Source files often contain rows that do not match the schema. Here it is a CSV row with `registration_date = NOT_A_DATE`. You can handle such rows in two different ways:

| Option | Bad row in the DataFrame? | Where the bad data goes |
|---|---|---|
| `mode = PERMISSIVE` (default) + `columnNameOfCorruptRecord` | **Yes**: unparseable fields become `NULL`, and `_corrupt_record` keeps the raw line | Stays in the DataFrame. Route it with `_corrupt_record IS NOT NULL` |
| `badRecordsPath` (Databricks) | **No**: the row is removed from the result | JSON files under `badRecordsPath/<timestamp>/bad_records/` with `path`, `record`, `reason` |

With `badRecordsPath` set, bad rows never reach the DataFrame, so a `_corrupt_record` column would always be `NULL`. Use one approach or the other.

In [0]:
# Create a CSV with a bad row
bad_csv_data = [
    (999, "Eve", "2023-01-03"),
    (888, "Frank", "NOT_A_DATE") # This will fail date parsing
]
(spark.createDataFrame(bad_csv_data, ["customer_id", "first_name", "registration_date"])
    .write.mode("overwrite").option("header", "true")
    .csv(f"{BATCH_SOURCE_PATH}/bad_csv"))

Option 1 is PERMISSIVE mode with `_corrupt_record`. Spark returns every row, and the bad one keeps its raw text, so you can route it later.

In [0]:
# Option 1: keep every row and the raw text of the bad one
df_permissive = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("mode", "PERMISSIVE")                           # default: keep every row
    .option("columnNameOfCorruptRecord", "_corrupt_record")  # must also be declared in the schema
    .schema("""
        customer_id STRING,
        first_name STRING,
        registration_date DATE,
        _corrupt_record STRING
    """)
    .load(f"{BATCH_SOURCE_PATH}/bad_csv")
)
display(df_permissive)   # Frank: registration_date = NULL, _corrupt_record = raw CSV line

In the result, look at Frank: `registration_date` is `NULL` and `_corrupt_record` holds the raw CSV line.

Option 2 is `badRecordsPath`. Spark removes the bad row from the result and writes it to files, so it never reaches the DataFrame.

In [0]:
# Option 2: send bad rows to files under BAD_RECORDS_PATH
df_with_errors = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("badRecordsPath", BAD_RECORDS_PATH)   # bad rows go to files, not to the DataFrame
    .schema("""
        customer_id STRING,
        first_name STRING,
        registration_date DATE
    """)
    .load(f"{BATCH_SOURCE_PATH}/bad_csv")
    .withColumn("_ingestion_timestamp", F.current_timestamp())
)
display(df_with_errors)   # only Eve — Frank was written to BAD_RECORDS_PATH

Only Eve is left in the DataFrame. The next two cells read the bad records statistics that Spark wrote for Frank.

In [0]:
# One glob per <timestamp> folder under BAD_RECORDS_PATH
files = [f.path for f in dbutils.fs.ls(BAD_RECORDS_PATH)]
files = [f + "*" for f in files]
print(files)

In [0]:
# Read the bad record files: each row has path, record and reason
from pyspark.sql.types import StructType, StructField, StringType

bad_record_schema = StructType([
    StructField("path", StringType(), True),
    StructField("record", StringType(), True),
    StructField("reason", StringType(), True)
])

df_bad_records = (
    spark.read
    .format("json")
    .schema(bad_record_schema)
    .load(files)
)
display(df_bad_records)

The result shows Frank's row with its `path`, `record` and `reason`.

## Change Data Feed for Incremental ETL

> **Summary** (see [Use change data feed](https://learn.microsoft.com/azure/databricks/tables/features/change-data-feed)): change data feed records row-level changes between versions of a Delta table. Each change row has a `_change_type` (`insert`, `update_preimage`, `update_postimage`, `delete`). You read the changes with `table_changes('table_name', version)` or the `readChangeFeed` option, so incremental ETL does not reprocess the whole table.

<a href="$./03_delta_lake">03 — Delta Lake</a> showed how to enable CDF and read change data with the `_change_type` metadata. In an efficient incremental ETL pipeline, each run uses CDF to process **only the rows that changed** since the last run, instead of the whole table.

### Example: Incremental ETL with CDF

The goal is an incremental pipeline that reads only the changed rows from a CDF-enabled source table. The steps:

1. Track the `last_processed_version` (checkpoint).
2. Read the changes committed **after** that version (`startingVersion = last_processed_version + 1`) with `readChangeFeed`.
3. Keep `insert` and `update_postimage` rows, and skip pre-images and deletes.
4. Keep the latest change per key (`_commit_version`), apply the transformations and write to the target table.

In [ ]:
# [1/2] Create a fresh CDF-enabled source table for this demo
# Own name, so cdf_demo from 03_delta_lake stays untouched and re-runs start from a clean history
TABLE_CDF = f"{CATALOG}.{BRONZE_SCHEMA}.cdf_stream_demo"
spark.sql(f"DROP TABLE IF EXISTS {TABLE_CDF}")
spark.sql(f"""
CREATE TABLE {TABLE_CDF} (
    user_id STRING, name STRING, email STRING,
    status STRING, updated_at TIMESTAMP
) USING DELTA
TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")
spark.sql(f"""
INSERT INTO {TABLE_CDF} VALUES
    ('U001', 'Alice', 'alice@example.com', 'premium', current_timestamp()),
    ('U003', 'Charlie', 'charlie@example.com', 'active', current_timestamp()),
    ('U004', 'Diana', 'diana@example.com', 'trial', current_timestamp())
""")

In [ ]:
# [2/2] Baseline: a pipeline run has processed everything up to the current version
last_processed_version = spark.sql(f"DESCRIBE HISTORY {TABLE_CDF}").agg(F.max("version")).first()[0]
print(f"last_processed_version = {last_processed_version}  (stored as the pipeline checkpoint)")
display(spark.table(TABLE_CDF))

In [ ]:
# Simulate new changes to the source table
spark.sql(f"""
INSERT INTO {TABLE_CDF} VALUES
    ('U005', 'Eve', 'eve@example.com', 'active', current_timestamp())
""")
spark.sql(f"""
UPDATE {TABLE_CDF}
SET status = 'active', updated_at = current_timestamp()
WHERE user_id = 'U004'
""")

In [ ]:
# [1/2] Read CDF: only commits after the last processed version
incremental_changes = (spark.read
    .format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", last_processed_version + 1)
    .table(TABLE_CDF))

display(incremental_changes
    .select("user_id", "name", "status", "_change_type", "_commit_version")
    .orderBy("_commit_version", "user_id", "_change_type"))

In [ ]:
# [2/2] Filter and transform: keep only current-state rows, skip pre-images and deletes
from pyspark.sql.window import Window

latest_per_key = Window.partitionBy("user_id").orderBy(F.col("_commit_version").desc())

transformed = (incremental_changes
    .filter(F.col("_change_type").isin(["insert", "update_postimage"]))
    # a key can change twice between runs, so keep the newest change
    .withColumn("_rn", F.row_number().over(latest_per_key))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
    .withColumn("processed_at", F.current_timestamp())
    .withColumn("email_domain", F.split(F.col("email"), "@")[1]))

display(transformed.select(
    "user_id", "name", "email_domain", "status",
    "_change_type", "_commit_version", "processed_at"
))

next_version = incremental_changes.agg(F.max("_commit_version")).first()[0]
print(f"Next run: store last_processed_version = {next_version}")

The result holds only the rows that changed after the baseline version, and the last line prints the version to store for the next run.

Choose the incremental method by its source:

| Method | Source | Best For |
|--------|--------|----------|
| COPY INTO | Files in cloud storage | Batch file ingestion with idempotency |
| Auto Loader | Files in cloud storage | Streaming file ingestion with schema evolution |
| CDF | Delta tables | Processing row-level changes between Delta tables |

> **Exam Note:** CDF is a **Delta-to-Delta** incremental method. For file-based ingestion, use COPY INTO or Auto Loader.

← <a href="$./04_incremental_ingestion">04 — Incremental Ingestion</a> | **<a href="$../../..">Course folder (README)</a>**

## Self-check (trainer): participants can skip this cell

In [ ]:
# Self-check (trainer): checks the values the demo cells saved and the surviving tables

# Stream-static join: customer_id repeats, yet every landed order gives one joined row
assert dup_ids > 0, "expected duplicated customer_id values in customers.csv"
landed_1_3 = spark.read.json(f"{STREAM_SOURCE_PATH}/batch_0[1-3]").count()
joined_rows = spark.table(TABLE_JOINED).count()
assert joined_rows == landed_1_3, f"{joined_rows:,} joined rows vs {landed_1_3:,} landed orders"

# Streaming aggregation: landed = batches 1-5 with a valid event time
assert 0 < counted <= landed, f"counted {counted:,}, landed {landed:,}"

# badRecordsPath: Frank's row went to the bad record files
assert df_bad_records.count() > 0, "no bad record files under BAD_RECORDS_PATH"

# CDF: only the inserted U005 and the updated U004, and a newer version to store
changed_ids = sorted(r.user_id for r in transformed.select("user_id").collect())
assert changed_ids == ["U004", "U005"], changed_ids
assert next_version > last_processed_version, (last_processed_version, next_version)

print("Self-check passed")